# **Digital Twin**

In [2]:
import numpy as np
import cv2
from ultralytics import YOLO

## **Load trained YOLO and Detect pieces on normal image with YOLO**

In [3]:
# === Step 1: Load YOLO model and run inference ===

model = YOLO("runs/detect/train2/weights/best.pt")
image_path = "G000_IMG000.jpg"

# Load image
original_image = cv2.imread(image_path)
results = model(original_image)
detections = results[0]

# Get bounding boxes, class indices, and confidences
boxes = detections.boxes.xyxy.cpu().numpy()  # [x_min, y_min, x_max, y_max]
classes = detections.boxes.cls.cpu().numpy()  # class indices
confidences = detections.boxes.conf.cpu().numpy()


0: 1024x1024 8 white-pawns, 2 white-rooks, 2 white-knights, 2 white-bishops, 1 white-queen, 1 white-king, 8 black-pawns, 2 black-rooks, 2 black-knights, 2 black-bishops, 1 black-queen, 1 black-king, 1 chess-board, 81.3ms
Speed: 6.0ms preprocess, 81.3ms inference, 1.0ms postprocess per image at shape (1, 3, 1024, 1024)


## **Detect chessboard and get homography matrix H (code from task 1)**

In [4]:
# === Step 2: Detect and warp the chessboard ===

def transform_image(img):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gaussian_blur = cv2.GaussianBlur(gray_img, (5, 5), 0)

    sobel_x = cv2.Sobel(gaussian_blur, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gaussian_blur, cv2.CV_64F, 0, 1, ksize=3)
    sobel_edges = cv2.magnitude(sobel_x, sobel_y)
    sobel_edges = cv2.convertScaleAbs(sobel_edges)

    _, otsu_binary = cv2.threshold(sobel_edges, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_clean = np.ones((3, 3), np.uint8)
    otsu_clean = cv2.morphologyEx(otsu_binary, cv2.MORPH_OPEN, kernel_clean)

    canny = cv2.Canny(otsu_clean, 100, 255)

    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(canny, kernel, iterations=1)

    lines = cv2.HoughLinesP(dilated, 1, np.pi / 180, threshold=200, minLineLength=100, maxLineGap=50)

    line_image = np.zeros_like(dilated)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_image, (x1, y1), (x2, y2), 255, 2)

    line_image = cv2.dilate(line_image, kernel, iterations=3)
    return line_image, img, otsu_binary

def order_points(pts):
    center = np.mean(pts, axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    return pts[np.argsort(angles)].astype("float32")

def find_board_contour(binary_image, otsu_binary):
    contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    max_area, best_cnt, points = 0, None, 3

    while max_area <= 50000:
        points += 1
        if points > 8:
            points = 4
            kernel_clean = np.ones((4, 4), np.uint8)
            otsu_clean = cv2.morphologyEx(otsu_binary, cv2.MORPH_OPEN, kernel_clean)
            canny = cv2.Canny(otsu_clean, 100, 255)
            dilated = cv2.dilate(canny, np.ones((3, 3), np.uint8), iterations=1)

            lines = cv2.HoughLinesP(dilated, 1, np.pi / 180, 200, minLineLength=100, maxLineGap=50)
            fallback_img = np.zeros_like(dilated)
            if lines is not None:
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                    cv2.line(fallback_img, (x1, y1), (x2, y2), 255, 2)
            binary_image = cv2.dilate(fallback_img, np.ones((3, 3), np.uint8), iterations=3)
            contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 10000:
                continue
            epsilon = 0.05 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, epsilon, True)
            if len(approx) == points and area > max_area and not any(0 in pt for pt in approx.reshape(-1, 2)):
                max_area = area
                best_cnt = approx

    if best_cnt is not None:
        if len(best_cnt) != 4:
            rect = cv2.minAreaRect(best_cnt)
            box = cv2.boxPoints(rect)
            best_cnt = np.int0(box)
        return order_points(best_cnt.reshape(4, 2))
    return None

def transform_perspective(rect, image, size=(800, 800)):
    if rect is None:
        return None, None
    dst = np.array([[0, 0], [size[0]-1, 0], [size[0]-1, size[1]-1], [0, size[1]-1]], dtype="float32")
    H = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, H, size)
    return warped, H


In [5]:
# === Orientation helpers ===
def find_darkest_corner(image, margin=100):
    h, w = image.shape[:2]
    corners = {
        'top_left': image[0:margin, 0:margin],
        'top_right': image[0:margin, w-margin:w],
        'bottom_right': image[h-margin:h, w-margin:w],
        'bottom_left': image[h-margin:h, 0:margin]
    }
    averages = {name: np.mean(cv2.cvtColor(corner, cv2.COLOR_BGR2GRAY)) for name, corner in corners.items()}
    return min(averages, key=averages.get)

def rotate_to_bottom_left(image, matrix, origin_corner):
    if origin_corner == 'top_left':
        return [list(row) for row in zip(*matrix)][::-1], cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif origin_corner == 'top_right':
        return [row[::-1] for row in matrix[::-1]], cv2.rotate(image, cv2.ROTATE_180)
    elif origin_corner == 'bottom_right':
        return [list(row)[::-1] for row in zip(*matrix)], cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    return matrix, image  # already correct

In [6]:
# === Apply preprocessing & warping ===
binary_lines, color_image, otsu = transform_image(original_image)
rect = find_board_contour(binary_lines, otsu)
warped_board, H = transform_perspective(rect, color_image)

## **For each piece bounding box, find its center and warp it to top-down coords**

In [7]:
# === Step 3: Map YOLO center points using homography ===

warped_points = []
if H is not None:
    for (x_min, y_min, x_max, y_max), cls in zip(boxes, classes):
        x_center = (x_min + x_max) / 2
        y_center = (y_min + y_max) / 2
        pt = np.array([[[x_center, y_center]]], dtype=np.float32)
        warped_pt = cv2.perspectiveTransform(pt, H)
        warped_points.append((warped_pt[0][0][0], warped_pt[0][0][1], int(cls)))


## **Map warped points to board squares**

In [13]:
# === Step 4: Convert to grid square coordinates and build board matrix ===

board_matrix = [['' for _ in range(8)] for _ in range(8)]

SQUARE_SIZE = warped_board.shape[0] // 8 if warped_board is not None else 100

for x_w, y_w, cls in warped_points:
    col = int(x_w // SQUARE_SIZE)
    row = int(y_w // SQUARE_SIZE)
    if 0 <= row < 8 and 0 <= col < 8:
        if (results[0].names[cls] != "chess-board"):
            board_matrix[row][col] = results[0].names[cls]


In [14]:
# === Rotate board to make logo corner the bottom-left ===
if warped_board is not None:
    darkest = find_darkest_corner(warped_board)
    board_matrix, warped_board = rotate_to_bottom_left(warped_board, board_matrix, darkest)

In [15]:
# === print the board ===
for row in board_matrix:
    print(' '.join([piece or '.' for piece in row]))

. black-rook black-bishop black-queen black-king black-bishop black-knight black-rook
. black-pawn black-pawn black-pawn black-pawn black-pawn black-pawn black-pawn
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . white-pawn white-pawn white-pawn white-pawn white-pawn white-pawn
. white-pawn white-bishop white-queen white-king white-bishop white-knight white-rook
. . . . . . . .


## **Build JSON digital twin data from board_matrix**

In [16]:
from PIL import Image, ImageDraw, ImageFont

# Tamanho do tabuleiro
tile_size = 80
board_size = tile_size * 8

# Cores das casas
light_color = "#f0d9b5"
dark_color = "#b58863"

# Símbolos Unicode das peças
piece_symbols = {
    "white-pawn": "♙",
    "white-knight": "♘",
    "white-bishop": "♗",
    "white-rook": "♖",
    "white-queen": "♕",
    "white-king": "♔",
    "black-pawn": "♟",
    "black-knight": "♞",
    "black-bishop": "♝",
    "black-rook": "♜",
    "black-queen": "♛",
    "black-king": "♚",
}

# Criar imagem
img = Image.new("RGB", (board_size, board_size), "white")
draw = ImageDraw.Draw(img)

# Fonte (com suporte a Unicode)
try:
    font = ImageFont.truetype("DejaVuSans.ttf", tile_size - 10)
except IOError:
    font = ImageFont.load_default()

# Desenhar o tabuleiro e as peças
for row in range(8):
    for col in range(8):
        x0 = col * tile_size
        y0 = row * tile_size
        color = light_color if (row + col) % 2 == 0 else dark_color
        draw.rectangle([x0, y0, x0 + tile_size, y0 + tile_size], fill=color)

        piece = board_matrix[row][col]
        
        if piece:
            
            symbol = piece_symbols.get(piece, "?")
            bbox = font.getbbox(symbol)
            w = bbox[2] - bbox[0]
            h = bbox[3] - bbox[1]
            draw.text(
                (x0 + (tile_size - w) / 2, y0 + (tile_size - h) / 2),
                symbol,
                fill="black" if piece[:2]=="bl" else "white",
                font=font,
            )

# Guardar ou mostrar
img.save("chessboard_from_matrix.png")
img.show()